In [3]:
import clip
import torch
from torchvision.datasets import CocoCaptions
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
from string import punctuation
import json
import os

Matplotlib is building the font cache; this may take a moment.


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
data_root = '../datasets/SugarCrepe'

data_dict = {
    'add_obj'    : f'{data_root}/add_obj.json',
    'add_att'    : f'{data_root}/add_att.json',
    'replace_obj': f'{data_root}/replace_obj.json',
    'replace_att': f'{data_root}/replace_att.json',
    'replace_rel': f'{data_root}/replace_rel.json',
    'swap_obj'   : f'{data_root}/swap_obj.json',
    'swap_att'   : f'{data_root}/swap_att.json',
}

dataset = {}
for c, data_path in data_dict.items():
    dataset[c] = json.load(open(data_path, 'r', encoding='utf-8'))

# SugarCrepe used COCO 2017 validation set
coco_image_root = '/mnt/lustre/datasets/coco/images/val2017/'

In [5]:
def get_exact_negpos_text_emb(model, pos_captions, neg_captions, batch_size=512, normalize=False):
    '''
    Get exact text embeddings for positive and negative captions

    Args:
    - model: CLIP model
    - pos_captions: list of positive captions
    - neg_captions: list of negative captions
    - batch_size: batch size for encoding
    - normalize: whether to normalize the embeddings

    Returns:
    - text_features: positive text embeddings
    - neg_text_features: negative text embeddings
    '''
    model.eval()

    text_features = []
    neg_text_features = []

    with torch.no_grad():
        for i in tqdm(range(0, len(pos_captions), batch_size)):
            pos_batch = pos_captions[i:i+batch_size]
            neg_batch = neg_captions[i:i+batch_size]

            pos_batch = clip.tokenize(pos_batch).to(device)
            neg_batch = clip.tokenize(neg_batch).to(device)

            text_features.extend(model.encode_text(pos_batch))
            neg_text_features.extend(model.encode_text(neg_batch))

        text_features = torch.stack(text_features).float()
        neg_text_features = torch.stack(neg_text_features).float()

        if normalize:
            text_features /= text_features.norm(dim=-1, keepdim=True)
            neg_text_features /= neg_text_features.norm(dim=-1, keepdim=True)

        print(text_features.shape, neg_text_features.shape)

    return text_features, neg_text_features

In [6]:
def get_image_emb_batched(model, preprocess, device, images, batch_size=512, normalize=False):
    '''
    Get image embeddings for a list of images

    Args:
    - model: CLIP model
    - preprocess: image preprocessing function
    - device: device to run the model on
    - images: list of images
    - batch_size: batch size for encoding
    - normalize: whether to normalize the embeddings

    Returns:
    - image_features: image embeddings
    '''
    model.eval()

    image_features = []

    with torch.no_grad():
        for i in tqdm(range(0, len(images), batch_size)):
            batch_images = images[i:i+batch_size]
            image_input = torch.stack([preprocess(image).to(device) for image in batch_images])
            image_features.append(model.encode_image(image_input))

        image_features = torch.cat(image_features).squeeze().float()
        if normalize:
            image_features /= image_features.norm(dim=-1, keepdim=True)
        print(image_features.shape)

    return image_features

In [7]:
from learning_alignment import *

def sugarcrepe_eval(clip_version, model_path=None):
    '''
    Evaluate the SugarCrepe dataset

    Args:
    - clip_version: CLIP version
    - model_path: path to the linear alignment model
    '''

    model, preprocess = clip.load(clip_version, device=device)
    
    metrics = {}

    for c, data_dict in dataset.items():
        images = []
        pos_captions = []
        neg_captions = []

        for i, data in tqdm(data_dict.items(), desc=f'{c}'):
            image_path = os.path.join(coco_image_root, data['filename'])
            image = Image.open(image_path).convert('RGB')
            pos_caption = data['caption']
            neg_caption = data['negative_caption']

            images.append(image)
            pos_captions.append(pos_caption)
            neg_captions.append(neg_caption)


        print("Getting text embeddings")
        pos_text_embeddings, neg_text_embeddings = get_exact_negpos_text_emb(model, pos_captions, neg_captions, normalize=True)

        print("Getting image embeddings")
        image_embeddings = get_image_emb_batched(model, preprocess, device, images, normalize=True)
        
        print("Evaluation")

        if model_path:
            linear_model = CLIPAlignment(image_embeddings.shape[-1]).to(device)
            linear_model.load_state_dict(torch.load(model_path))
            linear_model.eval()
            pos_text_embeddings = linear_model(pos_text_embeddings)
            neg_text_embeddings = linear_model(neg_text_embeddings)

        pos_score = (image_embeddings * pos_text_embeddings).sum(dim=-1)
        neg_score = (image_embeddings * neg_text_embeddings).sum(dim=-1)

        correct = pos_score > neg_score
        metrics[c] = correct.sum().item()/len(data_dict)

    print()
    print((metrics['add_obj']+metrics['add_att'])/2)
    print((metrics['replace_obj']+metrics['replace_att']+metrics['replace_rel'])/3)
    print((metrics['swap_obj']+metrics['swap_att'])/2)

In [ ]:
sugarcrepe_eval('ViT-B/32')

In [ ]:
sugarcrepe_eval('ViT-B/32', 'coco_cache/coco_alignment_hnb.pt')